# Task 1 - Linear Regression Model (House Price Prediction)
**Name:** Krish Choudhary  
**Internship:** AI & ML Intern — Maincrafts Technology  
**Dataset:** California Housing Dataset  
**Date:** June 2025


## Step 1 - Importing Libraries

In [ ]:
# importing all the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("all imports done")

## Step 2 - Loading the Dataset

In [ ]:
# loading california housing dataset from sklearn
# this dataset has info about houses in california like avg rooms, income etc.
housing = fetch_california_housing(as_frame=True)

df = housing.data.copy()
df['MedHouseVal'] = housing.target  # target column = median house value

print("Dataset loaded successfully")
print("Shape:", df.shape)
df.head()

## Step 3 - Basic Data Exploration

In [ ]:
# checking basic info about the data
print("Columns:", list(df.columns))
print()
print(df.info())

In [ ]:
df.describe()

In [ ]:
# checking for null values - important step before training
print("Null values in each column:")
print(df.isnull().sum())
print()
print("No missing values found - data is clean!")

## Step 4 - Exploratory Data Analysis (EDA)

I explored the data to understand the distributions and relationships between features before jumping into modelling.

In [ ]:
# Distribution of the target variable
plt.figure(figsize=(8, 4))
plt.hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
plt.title('Distribution of Median House Value', fontsize=13)
plt.xlabel('Median House Value (in $100k)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# target variable is slightly right skewed with many values capped at 5.0 ($500k)

In [ ]:
# correlation heatmap to see how features relate to each other
plt.figure(figsize=(10, 7))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

# MedInc has the highest correlation with the target variable (~0.69)

In [ ]:
# scatter plot - MedInc vs target (strongest feature)
plt.figure(figsize=(7, 5))
plt.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.15, s=5, color='steelblue')
plt.xlabel('Median Income (in 10k USD)')
plt.ylabel('Median House Value ($100k)')
plt.title('Income vs House Value')
plt.tight_layout()
plt.show()

print("As income increases, house prices also tend to go up — makes sense!")

## Step 5 - Preparing Data for Training

In [ ]:
# separating features and target
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

print("Features:", list(X.columns))
print("Target: MedHouseVal")
print()

# splitting into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

## Step 6 - Training the Linear Regression Model

In [ ]:
# training the model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model training done!")
print()
print(f"Intercept: {model.intercept_:.4f}")
print()
print("Feature Coefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"  {col:15s} -> {coef:.5f}")

## Step 7 - Evaluating the Model

In [ ]:
# predicting on test data
y_pred = model.predict(X_test)

# calculating metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("===== Model Performance =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print()
print(f"So on average the model is off by around ${mae*100000:.0f} per prediction")
print(f"R2 score of {r2:.2f} means model explains {r2*100:.1f}% of the variance")

## Step 8 - Plotting Results

In [ ]:
# actual vs predicted plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=8, color='steelblue', label='predictions')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r-', linewidth=1.5, label='perfect fit line')
plt.xlabel('Actual Values ($100k)')
plt.ylabel('Predicted Values ($100k)')
plt.title('Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# residual plot - checking if errors are random
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred, residuals, alpha=0.2, s=7, color='coral')
axes[0].axhline(0, color='red', linewidth=1.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linewidth=1.5)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# residuals look mostly centered around 0 which is good
# slight spread at higher values suggests the model struggles with expensive houses

In [ ]:
# feature importance based on absolute coefficient values
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient')

plt.figure(figsize=(8, 5))
colors = ['coral' if c < 0 else 'steelblue' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## Step 9 - Saving the Model

In [ ]:
import pickle

# saving the trained model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved as model.pkl")

# quick test - predicting on a sample house
sample = pd.DataFrame([{
    'MedInc': 5.0, 'HouseAge': 20.0, 'AveRooms': 6.0,
    'AveBedrms': 1.2, 'Population': 500.0, 'AveOccup': 3.0,
    'Latitude': 34.0, 'Longitude': -118.0
}])
pred = model.predict(sample)[0]
print(f"Sample prediction: ${pred * 100000:.0f}")

## Summary & What I Learnt

**Model Results:**

| Metric | Value |
|--------|-------|
| MAE    | 0.4848 |
| RMSE   | 0.5862 |
| R²     | 0.6732 |

**Key Observations:**
- MedInc (Median Income) is by far the most important feature
- The model does reasonably well but struggles with high-value properties
- Residuals are mostly random which is a good sign

**What could make it better:**
- Try Ridge or Lasso regression with regularization
- Add polynomial features to capture non-linear patterns
- Use Random Forest or XGBoost which usually performs much better on this dataset
- Apply log transformation on skewed features like Population

Overall this was a great task to get familiar with the full ML pipeline — from loading data to saving the model!

**— Krish Choudhary | AI/ML Intern, Maincrafts Technology**
